# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khadeja-qureshi/Machine-Learning-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is unavailable.")

print("HF_TOKEN loaded successfully.")

HF_TOKEN loaded successfully.


In [13]:
%pip install -q duckdb pandas numpy

import os
import json
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(f"""
    CREATE SECRET hf_access (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH_TABLE = f"""
    read_parquet(
        '{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
    )
"""

print("Connected to March 2026 warehouse data.")

Connected to March 2026 warehouse data.


In [14]:
page_frame = con.sql(f"""
    WITH page_windows AS (
        SELECT
            client_hash_id,
            content_hash_id,

            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-15'
            ) AS feature_days_present,

            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-16'
                                      AND DATE '2026-03-31'
            ) AS outcome_days_present,

            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-15'
            ) AS impressions_first15,

            SUM(gsc_clicks) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-15'
            ) AS clicks_first15,

            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01'
                                         AND DATE '2026-03-15'
                     AND gsc_impressions > 0
                    THEN gsc_avg_position * gsc_impressions
                    ELSE 0
                END
            )
            /
            NULLIF(
                SUM(gsc_impressions) FILTER (
                    WHERE report_date BETWEEN DATE '2026-03-01'
                                          AND DATE '2026-03-15'
                ),
                0
            ) AS avg_position_first15,

            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-16'
                                      AND DATE '2026-03-31'
            ) AS impressions_outcome

        FROM {MARCH_TABLE}

        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        client_hash_id,
        content_hash_id,
        feature_days_present,
        outcome_days_present,
        impressions_first15,
        clicks_first15,

        100.0 * clicks_first15
            / NULLIF(impressions_first15, 0)
            AS ctr_pct_first15,

        avg_position_first15,

        impressions_first15
            / NULLIF(feature_days_present, 0)
            AS avg_daily_impressions_first15,

        impressions_outcome
            / NULLIF(outcome_days_present, 0)
            AS avg_daily_impressions_outcome

    FROM page_windows

    WHERE feature_days_present >= 10
      AND outcome_days_present >= 10
      AND impressions_first15 >= 100
""").df()

page_frame["daily_impression_loss"] = (
    page_frame["avg_daily_impressions_first15"]
    - page_frame["avg_daily_impressions_outcome"]
).clip(lower=0)

page_frame["declined_20pct"] = (
    page_frame["avg_daily_impressions_outcome"]
    < 0.80 * page_frame["avg_daily_impressions_first15"]
).astype(int)

print("Eligible pages:", f"{len(page_frame):,}")
print(
    "Observed decline rate:",
    round(page_frame["declined_20pct"].mean(), 3)
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible pages: 77,446
Observed decline rate: 0.328


In [15]:
required_columns = [
    "client_hash_id",
    "content_hash_id",
    "impressions_first15",
    "clicks_first15",
    "ctr_pct_first15",
    "avg_position_first15",
    "avg_daily_impressions_first15",
    "avg_daily_impressions_outcome",
    "declined_20pct",
]

missing_columns = [
    column
    for column in required_columns
    if column not in page_frame.columns
]

assert not missing_columns, f"Missing columns: {missing_columns}"

print("page_frame is ready.")
print("Rows:", f"{len(page_frame):,}")

page_frame is ready.
Rows: 77,446


In [16]:
# SIGNAL CHECK 1 — CTR BY SEARCH-POSITION BUCKET
# This is linked to FlyRank's CTR-fix logic.

position_conditions = [
    page_frame["avg_position_first15"] <= 3,
    page_frame["avg_position_first15"] <= 10,
    page_frame["avg_position_first15"] <= 20,
    page_frame["avg_position_first15"] <= 50,
]

position_labels = [
    "top_3",
    "page_1",
    "striking",
    "page_3_5",
]

page_frame["position_bucket"] = np.select(
    position_conditions,
    position_labels,
    default="deep",
)

position_order = [
    "top_3",
    "page_1",
    "striking",
    "page_3_5",
    "deep",
]

page_frame["position_bucket"] = pd.Categorical(
    page_frame["position_bucket"],
    categories=position_order,
    ordered=True,
)

signal_1_table = (
    page_frame
    .groupby("position_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        median_position=("avg_position_first15", "median"),
        mean_ctr_pct=("ctr_pct_first15", "mean"),
        median_ctr_pct=("ctr_pct_first15", "median"),
    )
    .round(3)
)

display(signal_1_table)

,n,median_position,mean_ctr_pct,median_ctr_pct
position_bucket,,,,
top_3,10116,2.190,0.382,0.234
page_1,38789,5.675,0.337,0.189
striking,13316,13.847,0.263,0.061
page_3_5,13516,29.162,0.150,0.000
deep,1709,61.443,0.042,0.000


### Signal 1 verdict: **CONFIRMED**

The table compares CTR across search-position buckets and includes the number of pages (`n`) in every bucket.

Observed mean CTR decreased consistently as average search position became worse, falling from **0.382%** in the `top_3` bucket to **0.042%** in the `deep` bucket. Median CTR also declined from **0.234%** to **0.000%**.

This signal is linked to FlyRank's CTR-fix logic. It supports evaluating CTR relative to search position rather than applying one universal CTR threshold to every page.

In [17]:
# SIGNAL CHECK 2 — IMPRESSION VOLUME AND LATER ABSOLUTE LOSS
# Outcome values are used only to audit the signal, not in the rule.

page_frame["daily_impression_loss"] = (
    page_frame["avg_daily_impressions_first15"]
    - page_frame["avg_daily_impressions_outcome"]
).clip(lower=0)

volume_rank = page_frame["impressions_first15"].rank(
    method="first"
)

page_frame["volume_bucket"] = pd.qcut(
    volume_rank,
    q=4,
    labels=[
        "low",
        "medium",
        "high",
        "very_high",
    ],
)

signal_2_table = (
    page_frame
    .groupby("volume_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        median_impressions=("impressions_first15", "median"),
        mean_daily_impression_loss=("daily_impression_loss", "mean"),
        median_daily_impression_loss=("daily_impression_loss", "median"),
        decline_rate=("declined_20pct", "mean"),
    )
    .round(3)
)

display(signal_2_table)

,n,median_impressions,mean_daily_impression_loss,median_daily_impression_loss,decline_rate
volume_bucket,,,,,
low,19362,154.0,1.871,0.00,0.330
medium,19361,368.0,4.414,0.20,0.334
high,19361,896.0,10.021,0.00,0.309
very_high,19362,3115.0,58.256,6.24,0.338


### Signal 2 verdict: **MIXED**

The table divides pages into four observed-impression buckets and includes the number of pages (`n`) in each bucket.

Mean daily impression loss increased strongly with visibility, rising from **1.873** in the low-volume bucket to **58.256** in the very-high-volume bucket. However, median loss was not consistently ordered, and the decline rate remained relatively similar across the four groups.

This means impression volume appears useful for measuring the possible size of an opportunity, but it does not show that high-volume pages are consistently more likely to decline.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

I confirm **Lane 2: Refresh / Content Opportunity Scoring**.

My baseline considers pages that:

- received at least 100 impressions during March 1–15;
- had an average search position between 1 and 20; and
- had a CTR below the median CTR of pages in the same search-position bucket.

The score combines:

- **70% CTR-deficit strength** — how far the page's CTR is below the median for its position bucket;
- **30% impression-volume strength** — how much observed search visibility the page has.

A higher score places the page closer to the top of the review queue.

The baseline outputs one reason code:

`HIGH_VISIBILITY_CTR_DEFICIT`

The action label is:

`REVIEW_FOR_REFRESH`

This action means that a human should inspect the title, search snippet, intent match and page content. It does not mean that the page should automatically be rewritten.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# STEP 1 — Expected CTR for each position bucket
# This uses only March 1–15 information.

expected_ctr_by_bucket = (
    page_frame
    .groupby(
        "position_bucket",
        observed=True,
    )["ctr_pct_first15"]
    .median()
)

display(
    expected_ctr_by_bucket.to_frame(
        "expected_ctr_pct"
    )
)

page_frame["expected_ctr_pct"] = (
    page_frame["position_bucket"]
    .map(expected_ctr_by_bucket)
    .astype(float)
)

# STEP 2 — CTR deficit

page_frame["ctr_deficit_ratio"] = (
    (
        page_frame["expected_ctr_pct"]
        - page_frame["ctr_pct_first15"]
    )
    /
    page_frame["expected_ctr_pct"].replace(
        0,
        np.nan,
    )
).clip(
    lower=0,
    upper=1,
).fillna(0)

# STEP 3 — Visibility strength
# Log transformation prevents very large pages from dominating completely.

volume_reference = (
    page_frame["impressions_first15"]
    .quantile(0.95)
)

page_frame["volume_strength"] = (
    np.log1p(page_frame["impressions_first15"])
    / np.log1p(volume_reference)
).clip(
    lower=0,
    upper=1,
)

# STEP 4 — One transparent baseline score

page_frame["baseline_score"] = (
    100
    * (
        0.70 * page_frame["ctr_deficit_ratio"]
        + 0.30 * page_frame["volume_strength"]
    )
).round(2)

# STEP 5 — Eligible queue

baseline_queue = page_frame[
    page_frame["avg_position_first15"].between(
        1,
        20,
        inclusive="both",
    )
    & (page_frame["ctr_deficit_ratio"] > 0)
].copy()

baseline_queue["reason_code"] = (
    "HIGH_VISIBILITY_CTR_DEFICIT"
)

baseline_queue["action_label"] = (
    "REVIEW_FOR_REFRESH"
)

baseline_queue = (
    baseline_queue
    .sort_values(
        [
            "baseline_score",
            "impressions_first15",
        ],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

baseline_queue.insert(
    0,
    "rank",
    range(1, len(baseline_queue) + 1),
)

# STEP 6 — Safe output columns only

QUEUE_COLUMNS = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "baseline_score",
    "reason_code",
    "action_label",
    "impressions_first15",
    "clicks_first15",
    "ctr_pct_first15",
    "expected_ctr_pct",
    "ctr_deficit_ratio",
    "avg_position_first15",
]

baseline_queue_output = baseline_queue[
    QUEUE_COLUMNS
].copy()

print(
    "Queue rows:",
    f"{len(baseline_queue_output):,}",
)

display(
    baseline_queue_output.drop(
        columns=[
            "client_hash_id",
            "content_hash_id",
        ]
    ).head(10)
)

# STEP 7 — Write the required CSV

os.makedirs(
    "work/outputs",
    exist_ok=True,
)

OUTPUT_PATH = (
    "work/outputs/baseline_action_score.csv"
)

baseline_queue_output.to_csv(
    OUTPUT_PATH,
    index=False,
)

print("Queue written to:", OUTPUT_PATH)


,expected_ctr_pct
position_bucket,
top_3,0.234284
page_1,0.189296
striking,0.060515
page_3_5,0.000000
deep,0.000000


Queue rows: 30,445


,rank,baseline_score,reason_code,action_label,impressions_first15,clicks_first15,ctr_pct_first15,expected_ctr_pct,ctr_deficit_ratio,avg_position_first15
0,1,100.0,HIGH_VISIBILITY_CTR_DEFICIT,REVIEW_FOR_REFRESH,27715.0,0.0,0.0,0.189296,1.0,4.448854
1,2,100.0,HIGH_VISIBILITY_CTR_DEFICIT,REVIEW_FOR_REFRESH,18554.0,0.0,0.0,0.234284,1.0,2.992886
2,3,100.0,HIGH_VISIBILITY_CTR_DEFICIT,REVIEW_FOR_REFRESH,18421.0,0.0,0.0,0.189296,1.0,5.168503
3,4,100.0,HIGH_VISIBILITY_CTR_DEFICIT,REVIEW_FOR_REFRESH,17150.0,0.0,0.0,0.189296,1.0,8.154636
4,5,100.0,HIGH_VISIBILITY_CTR_DEFICIT,REVIEW_FOR_REFRESH,16711.0,0.0,0.0,0.189296,1.0,9.255520
5,6,100.0,HIGH_VISIBILITY_CTR_DEFICIT,REVIEW_FOR_REFRESH,15012.0,0.0,0.0,0.189296,1.0,4.410605
6,7,100.0,HIGH_VISIBILITY_CTR_DEFICIT,REVIEW_FOR_REFRESH,13576.0,0.0,0.0,0.189296,1.0,6.481806
7,8,100.0,HIGH_VISIBILITY_CTR_DEFICIT,REVIEW_FOR_REFRESH,12634.0,0.0,0.0,0.189296,1.0,6.903514
8,9,100.0,HIGH_VISIBILITY_CTR_DEFICIT,REVIEW_FOR_REFRESH,12020.0,0.0,0.0,0.189296,1.0,6.724459
9,10,100.0,HIGH_VISIBILITY_CTR_DEFICIT,REVIEW_FOR_REFRESH,11830.0,0.0,0.0,0.189296,1.0,7.896196


Queue written to: work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top_20 = baseline_queue_output.head(20).copy()

visibility_median = (
    top_20["impressions_first15"].median()
)

top_20["confidence_note"] = np.select(
    [
        (
            top_20["ctr_deficit_ratio"] >= 0.75
        )
        & (
            top_20["impressions_first15"]
            >= visibility_median
        ),

        top_20["ctr_deficit_ratio"] >= 0.50,
    ],
    [
        (
            "Higher confidence: large CTR deficit "
            "and strong visibility."
        ),
        (
            "Moderate confidence: meaningful CTR deficit."
        ),
    ],
    default=(
        "Lower confidence: score is driven more by visibility."
    ),
)

display(
    top_20[
        [
            "rank",
            "baseline_score",
            "action_label",
            "reason_code",
            "confidence_note",
            "impressions_first15",
            "ctr_pct_first15",
            "expected_ctr_pct",
            "avg_position_first15",
        ]
    ]
)


,rank,baseline_score,action_label,reason_code,confidence_note,impressions_first15,ctr_pct_first15,expected_ctr_pct,avg_position_first15
0,1,100.0,REVIEW_FOR_REFRESH,HIGH_VISIBILITY_CTR_DEFICIT,Higher confidence: large CTR deficit and stron...,27715.0,0.0,0.189296,4.448854
1,2,100.0,REVIEW_FOR_REFRESH,HIGH_VISIBILITY_CTR_DEFICIT,Higher confidence: large CTR deficit and stron...,18554.0,0.0,0.234284,2.992886
2,3,100.0,REVIEW_FOR_REFRESH,HIGH_VISIBILITY_CTR_DEFICIT,Higher confidence: large CTR deficit and stron...,18421.0,0.0,0.189296,5.168503
3,4,100.0,REVIEW_FOR_REFRESH,HIGH_VISIBILITY_CTR_DEFICIT,Higher confidence: large CTR deficit and stron...,17150.0,0.0,0.189296,8.154636
4,5,100.0,REVIEW_FOR_REFRESH,HIGH_VISIBILITY_CTR_DEFICIT,Higher confidence: large CTR deficit and stron...,16711.0,0.0,0.189296,9.255520
5,6,100.0,REVIEW_FOR_REFRESH,HIGH_VISIBILITY_CTR_DEFICIT,Higher confidence: large CTR deficit and stron...,15012.0,0.0,0.189296,4.410605
6,7,100.0,REVIEW_FOR_REFRESH,HIGH_VISIBILITY_CTR_DEFICIT,Higher confidence: large CTR deficit and stron...,13576.0,0.0,0.189296,6.481806
7,8,100.0,REVIEW_FOR_REFRESH,HIGH_VISIBILITY_CTR_DEFICIT,Higher confidence: large CTR deficit and stron...,12634.0,0.0,0.189296,6.903514
8,9,100.0,REVIEW_FOR_REFRESH,HIGH_VISIBILITY_CTR_DEFICIT,Higher confidence: large CTR deficit and stron...,12020.0,0.0,0.189296,6.724459
9,10,100.0,REVIEW_FOR_REFRESH,HIGH_VISIBILITY_CTR_DEFICIT,Higher confidence: large CTR deficit and stron...,11830.0,0.0,0.189296,7.896196


In [21]:
def possible_wrong_reason(row):
    if row["avg_position_first15"] <= 3:
        return (
            "A featured snippet or another SERP feature may be "
            "absorbing clicks even though the page ranks highly."
        )

    if row["impressions_first15"] >= top_20[
        "impressions_first15"
    ].quantile(0.90):
        return (
            "One unusually high-volume query may dominate the "
            "page-level averages and make the recommendation look stronger."
        )

    if row["ctr_deficit_ratio"] >= 0.80:
        return (
            "The page may appear mainly for informational or zero-click "
            "queries where a low CTR is expected."
        )

    if row["avg_position_first15"] >= 15:
        return (
            "Average position may hide a mixture of strong and very weak "
            "query-level rankings."
        )

    return (
        "The 15-day window may reflect temporary seasonality or "
        "short-term search-result changes rather than a page problem."
    )


for _, row in top_20.iterrows():
    print(
        f"{int(row['rank'])}. "
        f"Action: {row['action_label']}. "
        f"Reason: {row['reason_code']}. "
        f"Confidence: {row['confidence_note']} "
        f"Why: {int(row['impressions_first15'])} impressions, "
        f"CTR {row['ctr_pct_first15']:.2f}% versus "
        f"{row['expected_ctr_pct']:.2f}% expected, "
        f"average position {row['avg_position_first15']:.1f}. "
        f"What would make it wrong: "
        f"{possible_wrong_reason(row)}"
    )

1. Action: REVIEW_FOR_REFRESH. Reason: HIGH_VISIBILITY_CTR_DEFICIT. Confidence: Higher confidence: large CTR deficit and strong visibility. Why: 27715 impressions, CTR 0.00% versus 0.19% expected, average position 4.4. What would make it wrong: One unusually high-volume query may dominate the page-level averages and make the recommendation look stronger.
2. Action: REVIEW_FOR_REFRESH. Reason: HIGH_VISIBILITY_CTR_DEFICIT. Confidence: Higher confidence: large CTR deficit and strong visibility. Why: 18554 impressions, CTR 0.00% versus 0.23% expected, average position 3.0. What would make it wrong: A featured snippet or another SERP feature may be absorbing clicks even though the page ranks highly.
3. Action: REVIEW_FOR_REFRESH. Reason: HIGH_VISIBILITY_CTR_DEFICIT. Confidence: Higher confidence: large CTR deficit and strong visibility. Why: 18421 impressions, CTR 0.00% versus 0.19% expected, average position 5.2. What would make it wrong: The page may appear mainly for informational or zer

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show possible weak picks from the top 20.
# These have comparatively smaller CTR deficits.

weak_pick_candidates = (
    top_20
    .sort_values(
        [
            "ctr_deficit_ratio",
            "baseline_score",
        ],
        ascending=[True, True],
    )
    .head(5)
)

display(
    weak_pick_candidates[
        [
            "rank",
            "baseline_score",
            "impressions_first15",
            "ctr_pct_first15",
            "expected_ctr_pct",
            "ctr_deficit_ratio",
            "avg_position_first15",
        ]
    ]
)

# Columns that would directly contain future or label-derived information
FORBIDDEN_EXACT_COLUMNS = {
    "avg_daily_impressions_outcome",
    "impressions_outcome",
    "declined_20pct",
    "daily_impression_loss",
    "is_declining",
    "leak_future_ratio",
}

# Existing product flags and unsafe future-oriented names
FORBIDDEN_SUBSTRINGS = [
    "future",
    "outcome",
    "declined",
    "impression_loss",
    "trend",
    "refresh_flag",
    "quick_win",
]

for column in baseline_queue_output.columns:
    assert column not in FORBIDDEN_EXACT_COLUMNS, (
        f"Forbidden leakage column found: {column}"
    )

    assert not any(
        term in column.lower()
        for term in FORBIDDEN_SUBSTRINGS
    ), f"Possible leakage or product flag found: {column}"

required_output_columns = {
    "baseline_score",
    "reason_code",
    "action_label",
}

assert required_output_columns.issubset(
    baseline_queue_output.columns
), "Required baseline output columns are missing."

print("Leakage check passed.")
print(
    "No future outcomes, label-derived inputs, "
    "or existing product flags appear in the queue."
)

,rank,baseline_score,impressions_first15,ctr_pct_first15,expected_ctr_pct,ctr_deficit_ratio,avg_position_first15
0,1,100.0,27715.0,0.0,0.189296,1.0,4.448854
1,2,100.0,18554.0,0.0,0.234284,1.0,2.992886
2,3,100.0,18421.0,0.0,0.189296,1.0,5.168503
3,4,100.0,17150.0,0.0,0.189296,1.0,8.154636
4,5,100.0,16711.0,0.0,0.189296,1.0,9.255520


Leakage check passed.
No future outcomes, label-derived inputs, or existing product flags appear in the queue.


### Weak pick 1 — Rank 12

Rank 12 may be a weak recommendation because its average position is **13.6** and its expected CTR is only **0.06%**. At that search depth, receiving no clicks during a 15-day period may be normal rather than evidence that the page requires a content refresh. A reviewer should inspect its query-level positions and search intent before making changes.

### Weak pick 2 — Rank 2

Rank 2 has **18,554 impressions**, an average position of approximately **3.0**, and no observed clicks. However, a featured snippet, knowledge panel, or another zero-click search feature may explain the result. The underlying queries and search-result layout should be checked before changing the page.

These weak picks show that the baseline is transparent but incomplete. It cannot observe query intent, search-result features, branded-query mix, seasonality, or actual page quality.

The leakage checks passed. The saved queue contains no future-window measurements, decline labels, label-derived inputs, or existing FlyRank product flags.

### Top-20 pattern

All twenty top-ranked pages had an observed CTR of **0.00%**. Their CTR-deficit ratios therefore reached the maximum or nearly the maximum value. As a result, impression volume became the main factor determining their order.

This is an important limitation of the baseline. It can identify highly visible pages receiving no clicks, but it cannot explain why they received no clicks. Query intent, zero-click search features, query-level ranking differences, and reporting conditions require human review.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.